# 🛰️ sentinel2-self-supervised · Self-supervised pretraining on Sentinel-2 vs training from scratch
 Open-data Sentinel-2 patches (Microsoft Planetary Computer) · masked-autoencoder (MAE) pretraining on **unlabelled** patches · frozen-encoder linear probe at **10 / 50 / 100 % labels** · spatially honest evaluation

| | |
|---|---|
| **How to run** | `Runtime → Run all`. Fully automatic: patches are fetched from Planetary Computer — no uploads, no keys, no Earth Engine login. First run downloads imagery (the slow part); re-runs use the cache in `data/interim`. |
| **Best runtime** | GPU (T4). CPU also works — pretraining is time-boxed, just slower. |
| **Modes** | `MODE = "demo"` (default) fits a free Colab session; set `MODE = "full"` in Block 1 for the reported run (more patches, longer pretraining budget). |
| **Outputs** | `reports/` (metrics with block-bootstrap 95 % CIs, learning curves, confusion matrices, data-quality report, pretraining log) + `models/mae_encoder.pth` + probe heads + auto-downloaded zip. |

**Pipeline:** 🛰️ unlabelled patches → 🧩 random masking → 🧠 MAE pretraining → 🏷️ WorldCover proxy labels → ❄️ frozen-encoder probe (10 / 50 / 100 %) → 📊 paired block-bootstrap comparison

**Design principles:** open data only · **no synthetic labels anywhere** · every number is computed in this run · the comparison is **paired and seeded** (both encoders are trained on the exact same labelled subsets) · the pipeline **stops or skips** rather than inventing data · it claims **nothing** the run does not show.

**How to read the scores:** downstream labels are **ESA WorldCover 2021 used as a proxy** (10 m raster, no field data). Every accuracy / F1 is *agreement with WorldCover*, not field-verified accuracy, and class 4 ("Peat bogs") is really WorldCover's herbaceous-wetland class. The encoder is **frozen** — this is a linear-probe comparison, which understates what full fine-tuning could achieve. Confidence intervals come from resampling whole 2.56 km test blocks.

**What this notebook does NOT claim:**
- that self-supervised pretraining **improves** over training from scratch — it *measures* the difference and reports whatever the data shows (the paired confidence interval may include 0, especially in demo mode);
- production-scale SSL: one small region, one season, a short time-boxed pretraining on a small ViT;
- any result beyond the label fractions, area and season shown here.

## Block 1 · Environment, config & reproducibility
**What it does:** installs only missing packages, imports everything, fixes all random seeds, switches PyTorch to deterministic kernels where available, detects CPU/GPU, and puts every setting in one `CFG` dictionary.

**Why it matters:** a fixed seed and a printed environment line make the run repeatable and auditable. GPU kernels are still not bit-exact, so expect small run-to-run differences. The exact Sentinel-2 scene IDs are recorded in `reports/run_info.json` so the same inputs can be re-fetched.

> 🎤 **Say to the panel:** *Every run is seeded and records its environment and the exact Sentinel-2 scene IDs, so a colleague can reproduce my numbers within run-to-run noise. All settings live in one dictionary — nothing is hidden in the code.*

In [ ]:
import sys, subprocess, importlib, time, os, json, glob, zipfile, shutil, copy, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # needed for deterministic cuBLAS

# ---- install only what is missing (fast on Colab) ----
_REQ = {"pystac_client": "pystac-client", "planetary_computer": "planetary-computer", "odc.stac": "odc-stac",
         "rasterio": "rasterio", "sklearn": "scikit-learn", "scipy": "scipy", "pyproj": "pyproj",
         "matplotlib": "matplotlib", "tqdm": "tqdm", "pandas": "pandas", "joblib": "joblib"}
def _missing():
    out = []
    for mod, pkg in _REQ.items():
        try: importlib.import_module(mod)
        except Exception: out.append(pkg)
    return out
_m = _missing()
if _m:
    print("Installing:", ", ".join(_m))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _m)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import rasterio
from rasterio.warp import reproject, Resampling
from pyproj import Transformer
from affine import Affine
import pystac_client, planetary_computer, odc.stac
from odc.geo.geobox import GeoBox
import dask
from dask.diagnostics import ProgressBar
from scipy import ndimage as ndi
from sklearn.ensemble import RandomForestClassifier  # noqa: F401  (kept for parity with the peatland pipeline utilities)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, cohen_kappa_score
import joblib
import torch, torch.nn as nn
from tqdm.auto import tqdm
import pandas as pd
from IPython.display import display, HTML

# ---- reproducibility ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
try: torch.use_deterministic_algorithms(True, warn_only=True)   # warn (not fail) if an op has no deterministic kernel
except Exception: pass
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count() or 2)
Path("reports").mkdir(exist_ok=True); Path("models").mkdir(exist_ok=True)
Path("data/interim").mkdir(parents=True, exist_ok=True); Path("data/raw").mkdir(parents=True, exist_ok=True)

# ---- presentation helpers (same style as the peatland notebook) ----
def banner(step, title, sub=""):
    display(HTML(f"""<div style="margin:28px 0 12px;padding:14px 20px;border-radius:10px;color:#fff;
      background:linear-gradient(90deg,#0f2027,#203a43,#2c5364);box-shadow:0 3px 8px rgba(0,0,0,.25);font-family:Segoe UI,Arial,sans-serif">
       <span style="background:#4ade80;color:#052e16;border-radius:50%;padding:3px 10px;font-weight:700;margin-right:10px">{step}</span>
       <span style="font-size:20px;font-weight:700">{title}</span>
       <div style="opacity:.8;margin-top:4px;font-size:13px">{sub}</div></div>"""))

def _note(msg, bg, fg, bar, icon):
    display(HTML(f'<div style="padding:9px 14px;margin:6px 0;background:{bg};color:{fg};border-left:5px solid {bar};'
                 f'border-radius:4px;font-family:Segoe UI,Arial,sans-serif;font-size:14px">{icon} {msg}</div>'))
def ok(m):   _note(m, "#dcfce7", "#14532d", "#22c55e", "✅")
def info(m): _note(m, "#e0f2fe", "#0c4a6e", "#0ea5e9", "ℹ️")
def warn(m): _note(m, "#fef3c7", "#78350f", "#f59e0b", "⚠️")

def html_table(rows, header, highlight=None, width="100%"):
    th = " ".join(f'<th style="padding:6px 10px;text-align:left;background:#1e293b;color:#fff">{h}</th>' for h in header)
    body = ""
    for i, r in enumerate(rows):
        bg = "#fef9c3" if highlight is not None and i == highlight else ("#f8fafc" if i % 2 else "#ffffff")
        body += "<tr>" + " ".join(f'<td style="padding:5px 10px;background:{bg};color:#0f172a">{c}</td>' for c in r) + "</tr>"
    display(HTML(f'<table style="border-collapse:collapse;width:{width};font-family:Segoe UI,Arial,sans-serif;font-size:13px;'
                 f'box-shadow:0 1px 4px rgba(0,0,0,.15)"><tr>{th}</tr>{body}</table>'))

TIMINGS = {}
class stage:
    """Context manager that records wall-time per pipeline stage."""
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): TIMINGS[self.name] = time.time() - self.t

def _py(o):
    """Make anything JSON-serialisable (numpy scalars/arrays, tuples...)."""
    if isinstance(o, dict): return {str(k): _py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_py(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray): return _py(o.tolist())
    if isinstance(o, float) and not np.isfinite(o): return None
    return o

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU) - Runtime > Change runtime type > T4 GPU makes pretraining much faster"
display(HTML(f"""<div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#ecfeff,#dcfce7);font-family:Segoe UI,Arial,sans-serif">
 <h2 style="margin:0;color:#064e3b">🛰️ sentinel2-self-supervised · MAE pretraining vs scratch · frozen-encoder probe</h2>
 <div style="margin-top:8px;color:#134e4a;font-size:14px">Pipeline: 🛰️ patches → 🧩 masking → 🧠 MAE pretraining → 🏷️ proxy labels → ❄️ linear probe → 📊 paired bootstrap</div>
 <div style="margin-top:6px;color:#475569;font-size:13px">Python {sys.version.split()[0]} · torch {torch.__version__} · CPU cores {os.cpu_count()} · GPU: {gpu} · seed {SEED}</div></div>"""))

In [ ]:
# ===================== ALL SETTINGS LIVE HERE (single source of truth) =====================
CFG = dict(
    year=2021, months=("05", "09"), max_cloud=30, max_scenes_per_tile=6,
    bands=["B02", "B03", "B04", "B08", "B11", "B12"], res=10, epsg=32629,
    patch=64, min_valid_ssl=0.98, max_ssl_patches=2000,
    ds_min_valid=1.00, ds_pure=0.90, max_lab_patches=1600,
    block_pat=4, split=(0.70, 0.15, 0.15), buffer_blocks=1,
    mae_dim=192, mae_depth=6, mae_heads=4, mae_patch=8, mae_mask=0.60,
    dec_dim=96, dec_depth=2, mae_lr=2e-4, mae_wd=0.05, mae_warmup=100,
    mae_batch=128, mae_max_epochs=120, mae_patience=3, mae_min_epochs=6,
    mae_budget_s=300 if torch.cuda.is_available() else 600,
    fractions=[0.10, 0.50, 1.00], probe_seeds=3, probe_C=[0.1, 1.0, 10.0],
    boot_n=1000, seed=SEED)
assert CFG["patch"] % CFG["mae_patch"] == 0, "patch size must be divisible by the ViT sub-patch size"

MODE = "demo"
if MODE == "demo":     # fast preset: fits a free Colab session end-to-end (first run = downloads dominate)
    CFG.update(max_ssl_patches=1600, max_lab_patches=1600, boot_n=500,
               mae_budget_s=240 if torch.cuda.is_available() else 480)
elif MODE == "full":   # the run to report
    CFG.update(max_ssl_patches=8000, max_lab_patches=5000, boot_n=1000,
               mae_budget_s=900 if torch.cuda.is_available() else 1800)

# two unlabelled pretraining areas + one geographically DISJOINT labelled area
SSL_AREAS = [dict(name="Irish Midlands bogs",   bbox=(-7.90, 53.10, -7.50, 53.40)),
             dict(name="Connemara blanket bog", bbox=(-9.90, 53.40, -9.50, 53.60))]
DS_AREA   = dict(name="Wicklow uplands & adjacent lowlands", bbox=(-6.40, 52.95, -6.00, 53.30))

CLASS_NAMES = ["Artificial", "Agriculture", "Forest", "Open semi-natural", "Peat bogs (proxy)", "Other wetlands", "Water"]
CLASS_COLORS = ["#d62728", "#ffd966", "#2e7d32", "#a8d08d", "#6d4c41", "#26a69a", "#1e88e5", "#ffffff"]
CMAP = ListedColormap(CLASS_COLORS)
PEAT = 4                     # WorldCover herbaceous wetland used as a peat PROXY (stated everywhere)
PS = 2                       # plotting stride (display only)

WC_LUT = np.full(256, 255, np.uint8)
for k, v in {50: 0, 40: 1, 10: 2, 20: 3, 30: 3, 60: 3, 100: 3, 90: 4, 95: 5, 80: 6}.items(): WC_LUT[k] = v

def _areas_disjoint(ssl_areas, ds_area):
    for a in ssl_areas:
        b1, b2 = a["bbox"], ds_area["bbox"]
        if not (b1[2] < b2[0] or b2[2] < b1[0] or b1[3] < b2[1] or b2[3] < b1[1]): return False
    return True
assert _areas_disjoint(SSL_AREAS, DS_AREA), "the labelled downstream area must be geographically disjoint from the SSL areas"

RUN_INFO = dict(ssl_areas=[], ds=None, mode=MODE)

banner("1", "Configuration", "one dictionary, no magic numbers hidden in the code")
html_table([["MODE", MODE]] +
           [[k, (", ".join(CFG[k]) if k == "bands" else str(v))] for k, v in CFG.items() if k != "bands"] +
           [["ssl areas", " | ".join(a["name"] for a in SSL_AREAS)], ["labelled area", DS_AREA["name"] + " (WorldCover 2021 proxy)"]],
           ["setting", "value"], width="70%")

## Block 2 · Unlabelled Sentinel-2 patches
**What it does:** for each SSL area, searches Planetary Computer for summer-`year` Sentinel-2 L2A scenes (≤ `max_cloud` % cloud), keeps the clearest scenes per MGRS tile, masks cloud/shadow with the SCL band **per scene**, builds a median composite on one 10 m grid, then cuts it into 64 × 64-pixel patches. Post-2022 processing baselines (+1000 DN offset) are detected and removed. Patches with < `min_valid_ssl` valid pixels are dropped; the few remaining gaps are filled with the patch median (**pretraining input only** — the labelled downstream patches in Block 5 require 100 % valid pixels). Everything is cached, so re-runs are fast.

**Why it matters:** self-supervised pretraining needs no labels — only imagery. Scene IDs and processing baselines are recorded for reproducibility, and the SSL areas are geographically disjoint from the labelled downstream area (Block 5), which removes one more route for information to leak between pretraining and evaluation.

> 🎤 **Say to the panel:** *Pretraining uses zero labels — just cloud-masked Sentinel-2 medians over two areas. Scene IDs are saved, so the exact inputs are reproducible, and the labelled evaluation area is somewhere else entirely.*

In [ ]:
banner("2", "Unlabelled Sentinel-2 patches", "cloud-masked per scene → median composite → 64×64 patches · cached · no labels involved")
catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

def make_geobox(bbox, frac=1.0, res=10, epsg=32629, block=64):
    """One 10 m grid for an area, trimmed so both dimensions are whole multiples of `block` (patches chip cleanly)."""
    tr = Transformer.from_crs(4326, epsg, always_xy=True)
    pts = [tr.transform(x, y) for x in (bbox[0], bbox[2]) for y in (bbox[1], bbox[3])]
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
    hw, hh = (max(xs) - min(xs)) / 2 * frac, (max(ys) - min(ys)) / 2 * frac
    xmin, ymin = np.floor((cx - hw) / res) * res, np.floor((cy - hh) / res) * res
    W = int((2 * hw) / res) // block * block
    H = int((2 * hh) / res) // block * block
    return GeoBox((H, W), Affine(res, 0, xmin, 0, -res, ymin + H * res), f"EPSG:{epsg}")

def gb_lonlat(gb):
    tr = Transformer.from_crs(gb.crs.to_wkt(), 4326, always_xy=True)
    H, W = tuple(gb.shape); a = gb.affine
    pts = [tr.transform(*(a * (c, r))) for c in (0, W) for r in (0, H)]
    return (min(p[0] for p in pts), min(p[1] for p in pts), max(p[0] for p in pts), max(p[1] for p in pts))

def pick_scenes(bbox, year):
    found = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox,
                           datetime=f"{year}-{CFG['months'][0]}-01/{year}-{CFG['months'][1]}-30",
                           query={"eo:cloud_cover": {"lt": CFG["max_cloud"]}}).item_collection()
    by = {}
    for it in found: by.setdefault(it.properties.get("s2:mgrs_tile", "?"), []).append(it)
    sel = []
    for t, its in by.items():
        its.sort(key=lambda i: i.properties.get("eo:cloud_cover", 100)); sel += its[:CFG["max_scenes_per_tile"]]
    return sel, {t: len(v) for t, v in by.items()}, len(found)

import hashlib
def _enc(a):  # float32 DN (with NaN) -> uint16 for compact caching
    return np.where(np.isfinite(a), np.clip(np.rint(a), 0, 65534), 65535).astype(np.uint16)
def _dec(u):
    a = u.astype("float32"); a[u == 65535] = np.nan; return a

def _cache_path(bbox, tag):
    key = hashlib.md5(json.dumps([list(bbox), {k: CFG[k] for k in ("year", "months", "max_cloud", "max_scenes_per_tile", "bands", "epsg", "res")}, tag],
                                 default=str).encode()).hexdigest()[:10]
    return Path("data/interim") / f"composite_{tag}_{key}.npz"

def build_composite(bbox, name, tag):
    """Cloud-masked median Sentinel-2 composite for one bbox -> (6, H, W) float32 DN (NaN = no clear obs) + provenance."""
    cache = _cache_path(bbox, tag)
    if cache.exists():
        _z = np.load(cache); meta = json.loads(str(_z["meta"]))
        info(f"<b>{name}</b>: composite loaded from cache ({meta['n_scenes']} scenes, baselines {meta['baselines']}). "
             "Delete <code>data/interim</code> to rebuild from Planetary Computer.")
        return _dec(_z["comp"]), meta, True
    sel, per_tile, n_found = pick_scenes(bbox, CFG["year"])
    if not sel: raise RuntimeError(f"No Sentinel-2 scenes found for {name} - widen the dates or cloud limit in CFG.")
    BASELINES = sorted({str(i.properties.get("s2:processing_baseline", "?")) for i in sel})
    def _ge4(b):
        try: return float(b) >= 4.0
        except ValueError: return None
    _flags = {_ge4(b) for b in BASELINES}
    OFFSET = 1000.0 if _flags == {True} else 0.0
    if len(_flags) > 1: warn(f"{name}: scenes mix processing baselines (some with the +1000 offset, some without); "
                             "a per-scene correction is NOT applied, so indices may be slightly biased.")
    gb = make_geobox(bbox, 1.0, CFG["res"], CFG["epsg"], block=CFG["patch"])
    ds = odc.stac.load(sel, bands=CFG["bands"] + ["SCL"], geobox=gb, groupby="solar_day",
                       chunks={"x": 1024, "y": 1024},
                       resampling={**{b: "bilinear" for b in CFG["bands"]}, "SCL": "nearest"}, fail_on_error=False)
    refl = ds[CFG["bands"]].to_array("band").astype("float32")
    good = ds["SCL"].isin([4, 5, 6])
    masked = refl.where(good & (refl > 0)) - OFFSET
    med = masked.chunk({"time": -1, "band": 1}).median("time", skipna=True)
    with ProgressBar():
        comp = med.values
    del ds, refl, good, masked, med
    meta = dict(n_scenes=len(sel), scene_ids=[i.id for i in sel], baselines=BASELINES, offset=OFFSET,
                n_found=int(n_found), per_tile={k: int(v) for k, v in per_tile.items()},
                grid=[int(v) for v in tuple(gb.shape)])
    np.savez(cache, comp=_enc(comp), meta=json.dumps(meta))
    return comp, meta, False

def chip_patches(comp, min_valid):
    """Cut a (6, H, W) composite into 64x64 patches; drop patches with too many gaps; fill leftover gaps with the patch median."""
    P = CFG["patch"]; H, W = comp.shape[1:]
    nR, nC = H // P, W // P
    patches, origins, valids = [], [], []
    for r in range(nR):
        for c in range(nC):
            ch = comp[:, r * P:(r + 1) * P, c * P:(c + 1) * P]
            v = float(np.isfinite(ch).mean())
            if v < min_valid: continue
            ok_ch = np.isfinite(ch).any((1, 2))
            if not ok_ch.all(): continue          # a band with no valid pixel at all -> drop the patch
            ch = ch.astype("float32").copy()
            for b in range(6):
                m = ~np.isfinite(ch[b])
                if m.any(): ch[b][m] = np.median(ch[b][np.isfinite(ch[b])])
            patches.append(np.clip(np.rint(ch), 0, 65534).astype(np.uint16))
            origins.append((r, c)); valids.append(v)
    return (np.stack(patches) if patches else np.zeros((0, 6, P, P), np.uint16)), np.asarray(origins, int), np.asarray(valids, float)

with stage("ssl_patches"):
    ssl_parts, ssl_origin_parts, ssl_valid_parts, ssl_area_idx = [], [], [], []
    _area_rows = []
    for ai, a_ in enumerate(SSL_AREAS):
        comp, meta, cached = build_composite(a_["bbox"], a_["name"], "ssl")
        gb = make_geobox(a_["bbox"], 1.0, CFG["res"], CFG["epsg"], block=CFG["patch"])
        pat, org, val = chip_patches(comp, CFG["min_valid_ssl"])
        ssl_parts.append(pat); ssl_origin_parts.append(org); ssl_valid_parts.append(val); ssl_area_idx.append(np.full(len(pat), ai))
        H, W = tuple(gb.shape)
        RUN_INFO["ssl_areas"].append(dict(name=a_["name"], bbox=list(a_["bbox"]), grid=[H, W],
                                          n_scenes=meta["n_scenes"], scene_ids=meta["scene_ids"],
                                          baselines=meta["baselines"], offset=meta["offset"], cached=cached, n_patches=int(len(pat))))
        _area_rows.append([a_["name"], f"{W*CFG['res']/1000:.1f} x {H*CFG['res']/1000:.1f} km", f"{meta['n_found']}", f"{meta['n_scenes']}",
                           f"{len(pat):,}", f"{val.mean()*100:.1f}%" if len(val) else "0"])
        del comp
    html_table(_area_rows,
               ["SSL area", "size", "scenes found", "scenes used", "patches kept", "mean patch validity"], width="80%")
    SSL_PATCHES = np.concatenate(ssl_parts) if ssl_parts else np.zeros((0, 6, CFG["patch"], CFG["patch"]), np.uint16)
    SSL_ORIGINS = np.concatenate(ssl_origin_parts) if ssl_origin_parts else np.zeros((0, 2), int)
    SSL_VALIDS = np.concatenate(ssl_valid_parts) if ssl_valid_parts else np.zeros((0,))
    SSL_AREA_IDX = np.concatenate(ssl_area_idx) if ssl_area_idx else np.zeros((0,), int)
    # cap the number of patches (seeded) so demo mode stays inside a free Colab session
    if len(SSL_PATCHES) > CFG["max_ssl_patches"]:
        rng = np.random.default_rng(CFG["seed"])
        keep = rng.choice(len(SSL_PATCHES), CFG["max_ssl_patches"], replace=False)
        SSL_PATCHES, SSL_ORIGINS, SSL_VALIDS, SSL_AREA_IDX = SSL_PATCHES[np.sort(keep)], SSL_ORIGINS[np.sort(keep)], SSL_VALIDS[np.sort(keep)], SSL_AREA_IDX[np.sort(keep)]
        info(f"Capped the patch set at <b>{CFG['max_ssl_patches']:,}</b> patches (seeded subsample) for MODE={MODE}; "
             f"set MODE = \"full\" in Block 1 to pretrain on more imagery.")
    np.savez("data/interim/ssl_patches.npz", patches=SSL_PATCHES, origins=SSL_ORIGINS, valids=SSL_VALIDS, area=SSL_AREA_IDX)
    if len(SSL_PATCHES) == 0:
        raise RuntimeError("No pretraining patches survived the validity filter - widen the SSL areas or lower CFG['min_valid_ssl']. "
                           "Stopping instead of inventing data.")
    ok(f"Pretraining patch set: <b>{len(SSL_PATCHES):,}</b> patches of {CFG['patch']}×{CFG['patch']} px × 6 bands "
       f"(~{SSL_PATCHES.nbytes/1e6:.0f} MB) · zero labels involved")

# ---- figure: a random sample of the unlabelled pretraining patches (true colour) ----
rng = np.random.default_rng(SEED + 11)
_show = rng.choice(len(SSL_PATCHES), min(8, len(SSL_PATCHES)), replace=False)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.8))
for axp, i in zip(axes.ravel(), _show):
    p_ = SSL_PATCHES[i].astype("float32") / 10000.0
    rgb_ = np.stack([np.clip(p_[2], 0, .35), np.clip(p_[1], 0, .35), np.clip(p_[0], 0, .35)], -1)  # B04,B03,B02
    rgb_ = (rgb_ - rgb_.min()) / (rgb_.max() - rgb_.min() + 1e-9)
    axp.imshow(rgb_, interpolation="nearest")
    axp.set_title(f"{SSL_AREAS[SSL_AREA_IDX[i]]['name'].split()[0]} · row {SSL_ORIGINS[i][0]}, col {SSL_ORIGINS[i][1]}", fontsize=9)
    axp.axis("off")
fig.suptitle("Sample of the UNLABELLED pretraining patches (true colour, Sentinel-2 median composite)", fontweight="bold")
plt.tight_layout(); plt.savefig("reports/01_ssl_patches.png", dpi=130); plt.show()
if len(SSL_PATCHES) < 400: warn("Fewer than 400 pretraining patches - MAE pretraining will be weak; widen the SSL areas or raise max_ssl_patches.")

## Block 3 · Augmentations, masking & input standardisation
**What it does:** prepares the three ingredients pretraining needs: (1) channel-wise standardisation of the reflectance values, (2) light **geometric** augmentation (random horizontal/vertical flips and 90° rotations — spectral values are never altered), and (3) the masked-autoencoder objective itself: each 64 × 64 patch is cut into 8 × 8-pixel sub-patches, **60 % of them are hidden**, and the model must reconstruct the hidden pixels from the remaining 40 %.

**Why it matters:** masking turns unlabelled imagery into a supervised-style training signal with **zero labels**. MAE is used instead of a contrastive method (SimCLR / MoCo) because it trains well with the small batches a free Colab session can afford; a colour-jitter-free augmentation keeps the spectral values honest. Everything below is seeded, so the same patches are masked the same way on a re-run.

> 🎤 **Say to the panel:** *The label signal is generated by hiding 60 % of each patch and asking the model to rebuild it — no human labels are involved anywhere in pretraining.*

In [ ]:
banner("3", "Augmentations, masking & input standardisation", "geometric aug only · 60 % of 8×8 sub-patches hidden per patch · the reconstruction target IS the label signal")

# ---- channel-wise standardisation, computed once from the pretraining patches ----
# transpose to (N, H, W, 6) BEFORE reshaping so the 6 values per row really are the 6 bands of one pixel
_flat = SSL_PATCHES.astype("float32").transpose(0, 2, 3, 1).reshape(-1, 6)          # DN
PATCH_MU = torch.tensor(_flat.mean(0) / 10000.0, dtype=torch.float32, device=DEVICE).view(1, 6, 1, 1)   # broadcast over (B, 6, H, W)
PATCH_SD = torch.tensor(_flat.std(0) / 10000.0 + 1e-6, dtype=torch.float32, device=DEVICE).view(1, 6, 1, 1)
del _flat
info("Input standardisation (per band, DN/10000): mean " + ", ".join(f"{v:.3f}" for v in PATCH_MU.reshape(-1).cpu().tolist()) +
     " · std " + ", ".join(f"{v:.3f}" for v in PATCH_SD.reshape(-1).cpu().tolist()))

def _batch_x(idx):
    """uint16 DN -> standardised float32 tensor on DEVICE."""
    x = torch.from_numpy(SSL_PATCHES[np.asarray(idx)].astype("float32")).to(DEVICE) / 10000.0
    return (x - PATCH_MU) / PATCH_SD

# ---- augmentation: flips + 90-degree rotations only ----
_AUG_GEN = torch.Generator().manual_seed(SEED + 55)
def augment_batch(x):
    B = x.shape[0]
    k = torch.randint(0, 4, (B,), generator=_AUG_GEN)
    fh = torch.randint(0, 2, (B,), generator=_AUG_GEN).bool()
    fv = torch.randint(0, 2, (B,), generator=_AUG_GEN).bool()
    out = x.clone()
    for i in range(B):
        t = out[i]
        if int(k[i]) > 0: t = torch.rot90(t, int(k[i]), dims=(1, 2))
        if fh[i]: t = torch.flip(t, dims=(2,))
        if fv[i]: t = torch.flip(t, dims=(1,))
        out[i] = t
    return out

# ---- MAE random masking (defined here, reused by the model in Block 4) ----
def random_token_mask(B, L, ratio, device):
    """Returns (ids_keep, ids_restore, mask, len_keep) - the standard MAE shuffling trick, seeded by torch's global RNG."""
    len_keep = int(round(L * (1 - ratio)))
    noise = torch.rand(B, L, device=device)
    ids = torch.argsort(noise, dim=1)
    ids_keep, ids_restore = ids[:, :len_keep], torch.argsort(ids, dim=1)
    mask = torch.ones(B, L, device=device); mask[:, :len_keep] = 0
    mask = torch.gather(mask, 1, ids_restore)
    return ids_keep, ids_restore, mask, len_keep

# ---- figure: what the model actually sees (original vs 60 %-masked patches) ----
L_, P_ = (CFG["patch"] // CFG["mae_patch"]) ** 2, CFG["mae_patch"]
_demo = np.random.default_rng(SEED + 3).choice(len(SSL_PATCHES), 4, replace=False)
_xd = _batch_x(_demo)
_ids_keep, _ids_restore, _mask, _len_keep = random_token_mask(len(_demo), L_, CFG["mae_mask"], DEVICE)
_maskgrid = _mask.view(len(_demo), CFG["patch"] // P_, CFG["patch"] // P_).cpu().numpy()
fig, axes = plt.subplots(2, 4, figsize=(13, 6.8))
for j in range(4):
    p_ = SSL_PATCHES[_demo[j]].astype("float32")
    rgb_ = np.stack([np.clip(p_[2], 0, .35), np.clip(p_[1], 0, .35), np.clip(p_[0], 0, .35)], -1)
    rgb_ = (rgb_ - rgb_.min()) / (rgb_.max() - rgb_.min() + 1e-9)
    axes[0, j].imshow(rgb_, interpolation="nearest"); axes[0, j].axis("off"); axes[0, j].set_title("original", fontsize=9)
    rgb2_ = rgb_ * 0.25 + 0.5 * np.kron(1 - _maskgrid[j], np.ones((P_, P_)))[..., None]
    axes[1, j].imshow(np.clip(rgb2_, 0, 1), interpolation="nearest"); axes[1, j].axis("off")
    axes[1, j].set_title("encoder sees (grey = hidden)", fontsize=9)
fig.suptitle(f"Masking demo · each 64×64 patch → {L_} sub-patches of {P_}×{P_} px · {L_ - _len_keep} hidden ({CFG['mae_mask']*100:.0f} %), {int(_len_keep)} visible", fontweight="bold")
plt.tight_layout(); plt.savefig("reports/02_masking.png", dpi=130); plt.show()
ok(f"Masking: {L_} tokens per patch · {int(_len_keep)} visible ({(1-CFG['mae_mask'])*100:.0f} %) · aug = flips/rotations only, no colour jitter")

## Block 4 · Masked-autoencoder pretraining
**What it does:** trains a small ViT (dim 192, depth 6, 8 × 8 sub-patches) to reconstruct the hidden pixels; a shallow decoder does the reconstructing and is **discarded afterwards** — the encoder is the product. The loss is MSE on the masked sub-patches only, with targets normalised per sub-patch. Training is time-boxed (`mae_budget_s`) with early stopping on a 5 % validation split of the pretraining patches.

**Why it matters:** the time cap means this is a *time-boxed* SSL result on a free Colab, not an architecture bake-off — and the notebook says so whenever it matters. The validation split guards against over-fitting the reconstruction to a tiny patch set.

> 🎤 **Say to the panel:** *The encoder is what matters; the decoder exists only to shape it. I time-box training, early-stop on a validation split, and I report whether the budget was hit.*

In [ ]:
banner("4", "Masked-autoencoder pretraining · model", "small ViT encoder + shallow decoder · decoder is discarded after pretraining")
class PatchEmbed(nn.Module):
    def __init__(self, in_ch, embed_dim, patch):
        super().__init__(); self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch, stride=patch)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)          # (B, L, D)

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Linear(int(dim * mlp_ratio), dim))
    def forward(self, x):
        h = self.norm1(x); x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))

class MAE(nn.Module):
    """Compact masked autoencoder (He et al., 2022 style) for 6-band 64x64 Sentinel-2 patches."""
    def __init__(self, cfg, in_ch=6):
        super().__init__()
        self.P, self.D, self.mask_ratio, self.in_ch = cfg["mae_patch"], cfg["mae_dim"], cfg["mae_mask"], in_ch
        self.grid = cfg["patch"] // self.P; self.L = self.grid ** 2
        self.embed = PatchEmbed(in_ch, self.D, self.P)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.L, self.D))
        self.blocks = nn.ModuleList([Block(self.D, cfg["mae_heads"]) for _ in range(cfg["mae_depth"])])
        self.norm = nn.LayerNorm(self.D)
        self.enc_to_dec = nn.Linear(self.D, cfg["dec_dim"])
        self.mask_token = nn.Parameter(torch.zeros(1, 1, cfg["dec_dim"]))
        self.dec_pos_embed = nn.Parameter(torch.zeros(1, self.L, cfg["dec_dim"]))
        self.dec_blocks = nn.ModuleList([Block(cfg["dec_dim"], 3) for _ in range(cfg["dec_depth"])])
        self.dec_norm = nn.LayerNorm(cfg["dec_dim"])
        self.pred = nn.Linear(cfg["dec_dim"], self.P * self.P * in_ch)
        nn.init.trunc_normal_(self.pos_embed, std=0.02); nn.init.trunc_normal_(self.dec_pos_embed, std=0.02)
        self.apply(self._init_w)
    @staticmethod
    def _init_w(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
    def patchify(self, x):
        B, C, H, W = x.shape; p = self.P
        return x.reshape(B, C, H // p, p, W // p, p).permute(0, 2, 4, 3, 5, 1).reshape(B, self.L, p * p * C)
    def unpatchify(self, t, B):
        g, p, C = self.grid, self.P, self.in_ch
        return t.reshape(B, g, g, p, p, C).permute(0, 5, 1, 3, 2, 4).reshape(B, C, g * p, g * p)
    def forward_encoder(self, x, ids_keep):
        x = self.embed(x) + self.pos_embed
        x = torch.gather(x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, x.shape[-1]))
        for b in self.blocks: x = b(x)
        return self.norm(x)
    def forward_decoder(self, x, ids_restore, B):
        Dd = self.dec_pos_embed.shape[-1]
        x = self.enc_to_dec(x)
        x = torch.cat([x, self.mask_token.expand(B, ids_restore.shape[1] - x.shape[1], Dd)], dim=1)
        x = torch.gather(x, 1, ids_restore.unsqueeze(-1).expand(-1, -1, Dd)) + self.dec_pos_embed
        for b in self.dec_blocks: x = b(x)
        return self.pred(self.dec_norm(x))
    def forward(self, x):
        B = x.shape[0]
        ids_keep, ids_restore, mask, _ = random_token_mask(B, self.L, self.mask_ratio, x.device)
        tgt = self.patchify(x).view(B, self.L, self.P, self.P, self.in_ch)
        mu_ = tgt.mean(dim=(2, 3), keepdim=True); sd_ = tgt.std(dim=(2, 3), keepdim=True) + 1e-6   # per-sub-patch normalisation
        tgt_n = ((tgt - mu_) / sd_).reshape(B, self.L, -1)
        pred = self.forward_decoder(self.forward_encoder(x, ids_keep), ids_restore, B)
        loss_tok = ((pred - tgt_n) ** 2).mean(dim=-1) * mask                                      # masked tokens only
        return loss_tok.sum() / mask.sum(), pred, mask, (mu_, sd_)
    @torch.no_grad()
    def encode(self, x):
        """Frozen-encoder features: final-norm token embeddings, mean over tokens -> (B, D)."""
        ids = torch.arange(self.L, device=x.device).unsqueeze(0).expand(x.shape[0], self.L)
        return self.forward_encoder(x, ids).mean(1)

torch.manual_seed(SEED)
MAE_MODEL = MAE(CFG).to(DEVICE)
N_PARAMS = sum(p.numel() for p in MAE_MODEL.parameters())
html_table([["encoder", f"ViT · dim {CFG['mae_dim']} · depth {CFG['mae_depth']} · heads {CFG['mae_heads']} · {CFG['mae_patch']}×{CFG['mae_patch']} sub-patches"],
            ["decoder", f"dim {CFG['dec_dim']} · depth {CFG['dec_depth']} (discarded after pretraining)"],
            ["mask ratio", f"{CFG['mae_mask']*100:.0f} % hidden per patch"],
            ["parameters (total)", f"{N_PARAMS/1e6:.2f} M"],
            ["optimiser", f"AdamW · lr {CFG['mae_lr']} · wd {CFG['mae_wd']} · warmup {CFG['mae_warmup']} steps"],
            ["time budget", f"{CFG['mae_budget_s']} s · max epochs {CFG['mae_max_epochs']} · patience {CFG['mae_patience']}"]], ["component", "setting"], width="70%")

In [ ]:
banner("4", "MAE pretraining", f"time-boxed at {CFG['mae_budget_s']} s · early stopping on a 5 % validation split of the pretraining patches")
rng = np.random.default_rng(SEED + 7)
_perm0 = rng.permutation(len(SSL_PATCHES))
_n_val = max(8, int(0.05 * len(SSL_PATCHES)))
va_idx, tr_idx = _perm0[:_n_val], _perm0[_n_val:]
info(f"Pretraining patches: {len(tr_idx):,} train · {len(va_idx)} validation (reconstruction loss only - no labels exist)")

opt = torch.optim.AdamW(MAE_MODEL.parameters(), lr=CFG["mae_lr"], weight_decay=CFG["mae_wd"])
hist, step, t0 = [], 0, time.time()
best_val, best_state, patience_left = float("inf"), None, CFG["mae_patience"]
budget_hit, stop_reason = False, "max epochs reached"
for epoch in range(1, CFG["mae_max_epochs"] + 1):
    MAE_MODEL.train()
    perm = torch.randperm(len(tr_idx), generator=torch.Generator().manual_seed(SEED + 1000 + epoch)).numpy()
    losses = []
    for s in range(0, len(tr_idx), CFG["mae_batch"]):
        x = augment_batch(_batch_x(tr_idx[perm[s:s + CFG["mae_batch"]]]))
        loss, *_ = MAE_MODEL(x)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        step += 1
        lr_scale = min(1.0, (step + 1) / CFG["mae_warmup"])                        # linear warmup, then constant
        for pg in opt.param_groups: pg["lr"] = CFG["mae_lr"] * lr_scale
        losses.append(float(loss.detach()))
    MAE_MODEL.eval(); vl = []
    with torch.no_grad():
        for s in range(0, len(va_idx), CFG["mae_batch"]):
            loss, *_ = MAE_MODEL(_batch_x(va_idx[s:s + CFG["mae_batch"]]))
            vl.append(float(loss))
    tr_loss, va_loss = float(np.mean(losses)), float(np.mean(vl))
    hist.append(dict(epoch=epoch, train_loss=round(tr_loss, 6), val_loss=round(va_loss, 6), elapsed_s=round(time.time() - t0, 1)))
    if va_loss < best_val - 1e-5:
        best_val, patience_left = va_loss, CFG["mae_patience"]
        best_state = copy.deepcopy(MAE_MODEL.state_dict())
    else: patience_left -= 1
    if patience_left == 0 and epoch >= CFG["mae_min_epochs"]: stop_reason = f"early stop (no val improvement for {CFG['mae_patience']} epochs)"; break
    if time.time() - t0 > CFG["mae_budget_s"]: budget_hit, stop_reason = True, "TIME BUDGET reached"; break
if best_state is not None: MAE_MODEL.load_state_dict(best_state)
PRETRAIN = dict(epochs_run=len(hist), best_val_loss=best_val, seconds=round(time.time() - t0, 1), budget_hit=budget_hit, stop_reason=stop_reason, params_m=round(N_PARAMS / 1e6, 2))
torch.save(dict(config=_py(CFG), state_dict=best_state, patch_mu=PATCH_MU.cpu(), patch_sd=PATCH_SD.cpu(), **_py(PRETRAIN)), "models/mae_encoder.pth")
pd.DataFrame(hist).to_csv("reports/pretrain_log.csv", index=False)
ok(f"Pretrained in {PRETRAIN['seconds']:.0f} s · {PRETRAIN['epochs_run']} epochs · best val reconstruction loss {best_val:.4f} · stop: {stop_reason}")
if budget_hit: warn("The <b>time budget was hit</b> - this is a time-boxed pretraining run. Any pretrained-vs-scratch comparison is about THIS budget, not about the method's ceiling.")

# ---- figure: reconstruction quality on validation patches (the product is the encoder, not these images) ----
MAE_MODEL.eval()
_sel = va_idx[:4]
with torch.no_grad():
    x = _batch_x(_sel)
    _loss, pred, mask, (mu_, sd_) = MAE_MODEL(x)
    _B_ = x.shape[0]
    _den = pred.view(_B_, MAE_MODEL.L, MAE_MODEL.P, MAE_MODEL.P, MAE_MODEL.in_ch) * sd_ + mu_
    recon_std = _den.reshape(_B_, MAE_MODEL.L, -1)
    recon = MAE_MODEL.unpatchify(recon_std, _B_) * PATCH_SD + PATCH_MU                    # back to DN
    tok = MAE_MODEL.patchify(x).view(_B_, MAE_MODEL.L, MAE_MODEL.P, MAE_MODEL.P, MAE_MODEL.in_ch)
    vis = tok * (1 - mask).view(_B_, MAE_MODEL.L, 1, 1, 1)
    masked_in = MAE_MODEL.unpatchify(vis.reshape(_B_, MAE_MODEL.L, -1), _B_) * PATCH_SD + PATCH_MU
def _rgb(dn):
    a = np.clip(dn[[2, 1, 0]] / 10000.0, 0, .3).transpose(1, 2, 0)   # (3, H, W) -> (H, W, 3) for imshow
    return (a - a.min()) / (a.max() - a.min() + 1e-9)
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for j in range(4):
    mg = mask[j].view(MAE_MODEL.grid, MAE_MODEL.grid).cpu().numpy()
    grey = np.kron(mg, np.ones((MAE_MODEL.P, MAE_MODEL.P)))[..., None]
    panels = [(SSL_PATCHES[_sel[j]].astype("float32"), "original (DN)"),
              (masked_in[j].cpu().numpy(), "input: visible tokens (grey = hidden)"),
              (recon[j].cpu().numpy(), "reconstruction")]
    for i, (arr, ttl) in enumerate(panels):
        axp = axes[i, j]
        rgb = _rgb(arr)
        if i == 1: rgb = np.nan_to_num(np.where(grey, np.nan, rgb), nan=0.35)
        axp.imshow(np.clip(rgb, 0, 1), interpolation="nearest"); axp.axis("off")
        if j == 0: axp.set_ylabel(ttl, fontsize=9)
        if i == 0: axp.set_title(f"val patch #{j}", fontsize=9)
fig.suptitle(f"MAE reconstructions on held-out pretraining patches · val loss {best_val:.4f} · the ENCODER is the product, not these images", fontweight="bold")
plt.tight_layout(); plt.savefig("reports/03_reconstructions.png", dpi=130); plt.show()

fig, axp = plt.subplots(figsize=(7.5, 4.4))
axp.plot([h["epoch"] for h in hist], [h["train_loss"] for h in hist], label="train loss (masked MSE)", color="#2a9d8f")
axp.plot([h["epoch"] for h in hist], [h["val_loss"] for h in hist], label="val loss", color="#e76f51")
axp.set_xlabel("epoch"); axp.set_ylabel("masked-MSE loss"); axp.grid(alpha=.3); axp.legend()
axp.set_title(f"Pretraining curve · stopped after {len(hist)} epochs ({stop_reason})", fontweight="bold")
plt.tight_layout(); plt.savefig("reports/04_pretrain_curve.png", dpi=130); plt.show()

## Block 5 · Downstream labelled set (WorldCover proxy) & spatial split
**What it does:** builds a Sentinel-2 composite over a third, geographically **disjoint** area; warps ESA WorldCover 2021 onto the same 10 m grid to act as **proxy labels**; keeps only patches that are 100 % valid and ≥ 90 % one single class ("pure" patches — this limits label noise on class boundaries); then assigns whole 2.56 km blocks of patches to train / validation / test (70 / 15 / 15) with a one-block exclusion ring, searching seeds until every split contains several classes.

**Why it matters:** random patch splits would leak information through spatial autocorrelation and flatter both encoders; whole-block assignment plus a buffer keeps the test honest. WorldCover is a *proxy* — it is a modelled map, not field truth, and class 4 ("Peat bogs") is really its herbaceous-wetland class. Every score in this notebook is agreement with WorldCover.

> 🎤 **Say to the panel:** *Evaluation is by whole spatial block with a buffer, on patches the encoder never saw, in an area that has no overlap with the pretraining areas — and the labels are an openly documented proxy.*

In [ ]:
banner("5", "Downstream labelled set (WorldCover proxy)", "third, disjoint area · only 100 %-valid, >=90 %-pure patches · whole 2.56 km blocks to train/val/test with a buffer")
with stage("labelled_set"):
    DS_COMP, DS_META, ds_cached = build_composite(DS_AREA["bbox"], DS_AREA["name"], "ds")
    DSG = make_geobox(DS_AREA["bbox"], 1.0, CFG["res"], CFG["epsg"], block=CFG["patch"])
    H, W = tuple(DSG.shape)
    # ---- ESA WorldCover proxy labels, warped onto the SAME grid ----
    items = list(catalog.search(collections=["esa-worldcover"], bbox=gb_lonlat(DSG)).item_collection())
    if not items: raise RuntimeError("No WorldCover items found for the labelled area - check the bbox.")
    def _yr(it):
        s_ = str(it.properties.get("start_datetime") or it.datetime or "0000")
        return int(s_[:4]) if s_[:4].isdigit() else 0
    best = min({_yr(i) for i in items}, key=lambda y: abs(y - CFG["year"]))
    ds_wc = odc.stac.load([i for i in items if _yr(i) == best], bands=["map"], geobox=DSG, resampling="nearest")
    arr = np.asarray(ds_wc["map"].values); arr = arr[0] if arr.ndim == 3 else arr
    LAB_MAP = WC_LUT[arr.astype(np.uint8)]
    scheme = f"worldcover {best}"
    info(f"Labels: <b>ESA {scheme}</b> mapped through the fixed 10-class LUT - a PROXY, not field truth.")
    # ---- chips: 100 % valid, >= 90 % pure ----
    P = CFG["patch"]; nR, nC = H // P, W // P
    chips, ys, xys, purs = [], [], [], []
    for r in range(nR):
        for c in range(nC):
            if not np.isfinite(DS_COMP[:, r*P:(r+1)*P, c*P:(c+1)*P]).all(): continue      # strict: no gaps at all
            lp = LAB_MAP[r*P:(r+1)*P, c*P:(c+1)*P]
            if (lp == 255).any(): continue                                                # unmapped WorldCover code inside
            vals, cnts = np.unique(lp, return_counts=True)
            k = int(vals[int(np.argmax(cnts))]); purity = float(cnts.max() / cnts.sum())
            if purity < CFG["ds_pure"]: continue
            chips.append(np.clip(np.rint(DS_COMP[:, r*P:(r+1)*P, c*P:(c+1)*P]), 0, 65534).astype(np.uint16))
            ys.append(k); xys.append((r, c)); purs.append(purity)
    DS_PATCHES = np.stack(chips) if chips else np.zeros((0, 6, P, P), np.uint16)
    DS_Y = np.asarray(ys, np.int64); DS_ORIGINS = np.asarray(xys, int).reshape(-1, 2); DS_PURITY = np.asarray(purs, float)
    if len(DS_PATCHES) == 0:
        raise RuntimeError("No labelled patches passed the validity + purity filters - lower CFG['ds_pure'], raise CFG['max_cloud'] "
                           "or enlarge the labelled area. Stopping instead of inventing data.")
    if len(DS_PATCHES) > CFG["max_lab_patches"]:
        rng = np.random.default_rng(SEED + 1)
        keep = np.sort(rng.choice(len(DS_PATCHES), CFG["max_lab_patches"], replace=False))
        DS_PATCHES, DS_Y, DS_ORIGINS, DS_PURITY = DS_PATCHES[keep], DS_Y[keep], DS_ORIGINS[keep], DS_PURITY[keep]
        info(f"Capped the labelled set at <b>{CFG['max_lab_patches']:,}</b> patches (seeded subsample) for MODE={MODE}.")
    RUN_INFO["ds"] = dict(name=DS_AREA["name"], bbox=list(DS_AREA["bbox"]), grid=[H, W], wc_year=int(best),
                          n_scenes=DS_META["n_scenes"], scene_ids=DS_META["scene_ids"], baselines=DS_META["baselines"],
                          offset=DS_META["offset"], n_patches=int(len(DS_PATCHES)), cached=ds_cached,
                          purity_mean=float(DS_PURITY.mean()) if len(DS_PURITY) else 0.0)
    ok(f"Labelled patch set: <b>{len(DS_PATCHES):,}</b> patches · mean purity {DS_PURITY.mean()*100 if len(DS_PURITY) else 0:.1f} % · "
       f"labels = {scheme} (proxy)")
    del DS_COMP

In [ ]:
banner("5", "Spatial block split of the labelled patches", "whole 2.56 km blocks go to train / val / test · exclusion ring between them · seed chosen from labels only")
BP = CFG["block_pat"]
nBR, nBC = nR // BP, nC // BP                       # full blocks only; edge patches outside are excluded
nb = nBR * nBC
n_te = max(2, round(CFG["split"][2] * nb)); n_va = max(2, round(CFG["split"][1] * nb))

def assign(seed):
    rng = np.random.default_rng(seed); perm = rng.permutation(nb)
    flat = np.zeros(nb, np.int8)
    for k, i in enumerate(perm): flat[i] = 2 if k < n_te else (1 if k < n_te + n_va else 0)
    bs = flat.reshape(nBR, nBC)
    st = np.ones((3, 3), bool)
    near_test = ndi.binary_dilation(bs == 2, st, iterations=CFG["buffer_blocks"])
    near_val = ndi.binary_dilation(bs == 1, st, iterations=CFG["buffer_blocks"])
    bs[near_test & (bs != 2)] = -1      # exclusion ring: train/val blocks touching test
    bs[near_val & (bs == 0)] = -1       # exclusion ring: train blocks touching validation
    return bs

def split_patches(bs):
    br = DS_ORIGINS[:, 0] // BP; bc = DS_ORIGINS[:, 1] // BP
    inside = (br < nBR) & (bc < nBC)
    sp = np.where(inside, bs[np.clip(br, 0, nBR - 1), np.clip(bc, 0, nBC - 1)], -1)
    return sp.astype(np.int8)

def balance_ok(sp):
    kept = sp >= 0
    if not kept.any(): return False, None
    sh = tuple(float((sp == k).sum() / kept.sum()) for k in (0, 1, 2))
    cls = [int(len(np.unique(DS_Y[sp == k]))) for k in (0, 1, 2)]
    te_bl = int(np.unique(DS_ORIGINS[sp == 2] // BP, axis=0).shape[0]) if (sp == 2).any() else 0
    # prefer splits that keep enough TEST blocks for meaningful block-bootstrap CIs
    return (sh[0] >= 0.55 and sh[1] >= 0.10 and sh[2] >= 0.10 and all(c >= 2 for c in cls) and te_bl >= 6), (sh, cls)

chosen_seed, BS, SP, binfo = None, None, None, None
for trial in range(SEED, SEED + 300):
    bs_ = assign(trial); sp_ = split_patches(bs_)
    good_, bi_ = balance_ok(sp_)
    if good_: chosen_seed, BS, SP, binfo = trial, bs_, sp_, bi_; break
if SP is None:
    chosen_seed = SEED; BS = assign(chosen_seed); SP = split_patches(BS)
    binfo = balance_ok(SP)[1]
    warn("No perfectly balanced split found - using the default seed; check the QC gate below.")

_split_names = {0: "Train", 1: "Validation", 2: "Test", -1: "Excluded (buffer ring / grid edge)"}
_kept = max(int((SP >= 0).sum()), 1)
html_table([[ _split_names[k], f"{int((SP==k).sum()):,}", (f"{(SP==k).sum()/_kept*100:.1f} % of kept" if k >= 0 else "-"),
              (f"{len(np.unique(DS_Y[SP==k]))} of 7" if k >= 0 and (SP==k).any() else "-")] for k in (0, 1, 2, -1)],
           ["split", "patches (64×64 px)", "share", "classes present"], width="70%")
ok(f"Block-split seed {chosen_seed} (chosen so train, val and test each contain >= 2 classes - decided from labels only, never from model scores)")

s_ = PS
fig, axs = plt.subplots(1, 3, figsize=(17.5, 6.2))
# true colour is reloaded from the composite cache (DS_COMP was freed above to save RAM)
_zds = np.load(_cache_path(DS_AREA["bbox"], "ds")); _comp_ds = _dec(_zds["comp"])
_c = _comp_ds.astype("float32") / 10000.0
_rgb = np.stack([np.clip(_c[2], 0, .3), np.clip(_c[1], 0, .3), np.clip(_c[0], 0, .3)], -1)
_rgb = (_rgb - np.nanpercentile(_rgb, 2)) / (np.nanpercentile(_rgb, 98) - np.nanpercentile(_rgb, 2) + 1e-9)
axs[0].imshow(np.nan_to_num(_rgb)[::s_, ::s_], interpolation="nearest"); axs[0].axis("off"); del _comp_ds, _c, _zds
axs[0].set_title(f"Sentinel-2 composite · {DS_AREA['name']}", fontweight="bold", fontsize=10.5)
axs[1].imshow(np.where(LAB_MAP == 255, 7, LAB_MAP)[::s_, ::s_], cmap=CMAP, vmin=0, vmax=7, interpolation="nearest")
axs[1].set_title(f"7-class proxy labels ({scheme})", fontweight="bold", fontsize=10.5); axs[1].axis("off")
axs[1].legend(handles=[mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(7)],
              loc="lower left", fontsize=7.5, framealpha=.9)
split_img = BS + 1        # 0 = buffer / excluded (-1), 1 = train, 2 = validation, 3 = test
axs[2].imshow(split_img, cmap=ListedColormap(["#9ca3af", "#1f77b4", "#2ca02c", "#d62728"]), vmin=0, vmax=3, interpolation="nearest")
axs[2].set_title(f"Spatial split (1 cell = 1 patch, blocks of {BP}×{BP}) · seed {chosen_seed}", fontweight="bold", fontsize=10.5); axs[2].axis("off")
axs[2].legend(handles=[mpatches.Patch(color=c, label=l) for c, l in
                       zip(["#9ca3af", "#1f77b4", "#2ca02c", "#d62728"], ["Buffer / excluded", "Train", "Validation", "Test"])],
              loc="lower left", fontsize=7.5, framealpha=.9)
plt.tight_layout(); plt.savefig("reports/05_labelled_area.png", dpi=130); plt.show()

In [ ]:
banner("5", "Data-quality gate", "independent checks recomputed from the final arrays · pipeline stops if a critical check fails")
checks = []
def chk(name, passed, detail, critical=True):
    checks.append(dict(name=name, status="pass" if passed else ("fail" if critical else "warn"), detail=detail))

chk("Label source is openly documented (WorldCover PROXY, never field truth)", True, f"{scheme} · fixed 10-class LUT · peat class is herbaceous wetland", critical=False)
chk("SSL areas and labelled area are geographically disjoint", _areas_disjoint(SSL_AREAS, DS_AREA),
    "no bbox overlap between pretraining and evaluation areas")
chk("Every kept labelled patch was 100 % finite at chip time", len(DS_PATCHES) == 0 or True,
    "chips were only kept when all 6 bands were finite (checked before uint16 storage, which cannot hold gaps)")
chk("Enough labelled patches for a probe (>= 300)", len(DS_PATCHES) >= 300, f"{len(DS_PATCHES):,} patches")
chk("Enough labelled patches for stable CIs (>= 600)", len(DS_PATCHES) >= 600, f"{len(DS_PATCHES):,} patches", critical=False)
chk("Mean patch purity meets the threshold", len(DS_PURITY) == 0 or float(DS_PURITY.mean()) >= CFG["ds_pure"],
    f"mean purity {DS_PURITY.mean()*100 if len(DS_PURITY) else 0:.1f} % (threshold {CFG['ds_pure']*100:.0f} %)")
chk("Train, validation and test blocks all non-empty", all((BS == k).any() for k in (0, 1, 2)),
    f"{int((BS==0).sum())} / {int((BS==1).sum())} / {int((BS==2).sum())} blocks")
# independent re-check: no block lands in two splits
_block_consistent = True
for _br in range(nBR):
    for _bc in range(nBC):
        _m = (DS_ORIGINS[:, 0] // BP == _br) & (DS_ORIGINS[:, 1] // BP == _bc) & (SP >= 0)
        if _m.any() and len(np.unique(SP[_m])) > 1: _block_consistent = False
chk("No spatial block appears in two splits", _block_consistent, f"{nb} blocks checked")
_d_te = ndi.distance_transform_cdt(BS != 2, metric="chessboard")
min_gap_te = int(_d_te[BS == 0].min()) if (BS == 0).any() else -1
chk("Train blocks >= buffer distance from every test block", min_gap_te >= CFG["buffer_blocks"] + 1,
    f"minimum gap = {min_gap_te} blocks (required {CFG['buffer_blocks']+1})")
_d_va = ndi.distance_transform_cdt(BS != 1, metric="chessboard")
min_gap_va = int(_d_va[BS == 0].min()) if ((BS == 0).any() and (BS == 1).any()) else -1
chk("Train blocks >= buffer distance from every validation block", min_gap_va >= CFG["buffer_blocks"] + 1,
    f"minimum gap = {min_gap_va} blocks (required {CFG['buffer_blocks']+1})")
chk("At least 10 test blocks (needed for meaningful block-bootstrap CIs)", int((BS == 2).sum()) >= 10,
    f"{int((BS==2).sum())} test blocks", critical=False)
split_counts = {k: np.bincount(DS_Y[SP == k], minlength=7).tolist() for k in (0, 1, 2)}
for k in (0, 1, 2):
    _c = split_counts[k]
    chk(f"At least 2 classes present in {_split_names[k].lower()}", sum(1 for v in _c if v > 0) >= 2,
        ", ".join(f"{CLASS_NAMES[i]}: {v:,}" for i, v in enumerate(_c) if v > 0), critical=True)
missing_all = [CLASS_NAMES[i] for i in range(7) if all(split_counts[k][i] == 0 for k in (0, 1, 2))]
chk("No class is entirely absent from the labelled set", not missing_all,
    "absent everywhere: " + (", ".join(missing_all) if missing_all else "none"), critical=False)

icon = {"pass": "✅", "warn": "⚠️", "fail": "❌"}
html_table([[icon[c["status"]], c["name"], c["detail"]] for c in checks], [" ", "check", "result"])
status = "fail" if any(c["status"] == "fail" for c in checks) else ("warn" if any(c["status"] == "warn" for c in checks) else "pass")
json.dump(_py(dict(status=status, label_source=scheme, split_seed=int(chosen_seed), n_patches=int(len(DS_PATCHES)),
                   purity_mean=float(DS_PURITY.mean()) if len(DS_PURITY) else None,
                   blocks=dict(train=int((BS == 0).sum()), val=int((BS == 1).sum()), test=int((BS == 2).sum())),
                   checks=checks, class_counts_per_split={_split_names[k]: dict(zip(CLASS_NAMES, split_counts[k])) for k in (0, 1, 2)})),
          open("reports/data_quality.json", "w"), indent=2)
if status == "fail": raise AssertionError("Data-quality gate FAILED - see the table above and reports/data_quality.json")
ok(f"Quality gate: <b>{status.upper()}</b> · report saved to reports/data_quality.json")

## Block 6 · Frozen-encoder features & the paired probe comparison
**What it does:** freezes both encoders — the MAE-pretrained one and the **same architecture randomly initialised** (seeded) — turns every labelled patch into one feature vector (mean of the encoder's token outputs), and trains an identical logistic-regression probe on top of each: the same `C` grid chosen on the validation blocks, the same stratified labelled subsets at 10 / 50 / 100 %, the same three subsampling seeds. Both encoders consume the **exact same feature inputs** and differ only in their weights.

**Why it matters:** freezing the encoder and re-using the same probe protocol makes this a controlled experiment: any score difference comes from pretraining, not from probe capacity or sampling luck. (Full fine-tuning could change the picture — stated as a limitation in Block 8, not tested here.)

> 🎤 **Say to the panel:** *Both encoders get the same architecture, the same probe and the exact same labelled subsets — the only difference is pretraining, and the difference is tested with a paired bootstrap, not by eyeballing two numbers.*

In [ ]:
banner("6", "Frozen-encoder features", "same architecture, two weight sets: MAE-pretrained vs seeded random init · mean token pooling")
ENC_PRE = MAE_MODEL                    # pretrained in Block 4 (best-validation state)
torch.manual_seed(SEED + 101)
ENC_SCR = MAE(CFG).to(DEVICE)          # identical architecture, freshly seeded random init = "training from scratch"
ENC_PRE.eval(); ENC_SCR.eval()

@torch.no_grad()
def extract_features(model, patches_u16, bs=64, desc=""):
    outs = []
    for s in tqdm(range(0, len(patches_u16), bs), desc=desc, leave=False):
        x = torch.from_numpy(patches_u16[s:s + bs].astype("float32")).to(DEVICE) / 10000.0
        x = (x - PATCH_MU) / PATCH_SD
        outs.append(model.encode(x).float().cpu().numpy())
    return np.concatenate(outs) if outs else np.zeros((0, CFG["mae_dim"]), np.float32)

with stage("features"):
    F_PRE_RAW = extract_features(ENC_PRE, DS_PATCHES, desc="pretrained encoder")
    F_SCR_RAW = extract_features(ENC_SCR, DS_PATCHES, desc="scratch encoder")
assert F_PRE_RAW.shape == F_SCR_RAW.shape and len(F_PRE_RAW) == len(DS_PATCHES)

tr_rows = SP == 0
MU_PRE, SD_PRE = F_PRE_RAW[tr_rows].mean(0), F_PRE_RAW[tr_rows].std(0) + 1e-6     # train-block statistics only
MU_SCR, SD_SCR = F_SCR_RAW[tr_rows].mean(0), F_SCR_RAW[tr_rows].std(0) + 1e-6
F_PRE = ((F_PRE_RAW - MU_PRE) / SD_PRE).astype("float32")
F_SCR = ((F_SCR_RAW - MU_SCR) / SD_SCR).astype("float32")
_const = {nm: int((sd_ < 1e-8).sum()) for nm, sd_ in (("pretrained", F_PRE_RAW.std(0)), ("scratch", F_SCR_RAW.std(0)))}
html_table([["pretrained (MAE)", F_PRE_RAW.shape[1], f"{np.abs(F_PRE_RAW).mean():.4f}", _const["pretrained"]],
            ["scratch (random init)", F_SCR_RAW.shape[1], f"{np.abs(F_SCR_RAW).mean():.4f}", _const["scratch"]]],
           ["encoder", "feature dim", "mean |activation|", "near-constant dims"], width="70%")
if _const["scratch"] > 0: warn(f"{_const['scratch']} scratch-encoder feature dimensions are near-constant - expected for an untrained encoder; the probe handles it via standardisation.")

In [ ]:
banner("6", "Paired probe comparison", "same subsets · same protocol · same C grid · only the encoder weights differ")
y_all = DS_Y.astype(int)
va_rows, te_rows = SP == 1, SP == 2
y_tr, y_va, y_te = y_all[tr_rows], y_all[va_rows], y_all[te_rows]
LABS = list(range(7))

def stratified_subset(y, frac, seed):
    """Per-class stratified subset of row indices - computed ONCE per (fraction, seed) and shared by BOTH encoders."""
    rng = np.random.default_rng(seed); out = []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        k = len(idx) if frac >= 1.0 else max(1, int(round(frac * len(idx))))
        out.append(rng.choice(idx, min(k, len(idx)), replace=False))
    return np.sort(np.concatenate(out))

def fit_probe(F, y, C):
    return LogisticRegression(max_iter=1000, C=C, class_weight="balanced", solver="lbfgs").fit(F, y)

def score_pack(y_true, y_pred):
    return dict(accuracy=float(accuracy_score(y_true, y_pred)),
                macro_f1=float(f1_score(y_true, y_pred, labels=LABS, average="macro", zero_division=0)),
                kappa=float(cohen_kappa_score(y_true, y_pred, labels=LABS)))

# ---- choose C on the VALIDATION blocks at 100 % of labels (same grid for both encoders, smaller C wins ties) ----
C_STAR = {}
for nm, F in (("pretrained", F_PRE), ("scratch", F_SCR)):
    best = (-1.0, None)
    for C in CFG["probe_C"]:
        s = score_pack(y_va, fit_probe(F[tr_rows], y_tr, C).predict(F[va_rows]))["macro_f1"]
        if s > best[0] + 1e-9 or (abs(s - best[0]) <= 1e-9 and (best[1] is None or C < best[1])): best = (s, C)
    C_STAR[nm] = best[1]
info("Probe C chosen on validation blocks: " + " · ".join(f"{nm}: C={C_STAR[nm]}" for nm in ("pretrained", "scratch")))

# ---- the paired runs: identical subsets for both encoders ----
FEATS = {"pretrained": F_PRE, "scratch": F_SCR}
rows, preds0 = [], {}
with stage("probe"):
    for frac in CFG["fractions"]:
        for r in range(CFG["probe_seeds"]):
            sub = stratified_subset(y_tr, frac, seed=SEED + 2000 + r)          # ONE subset, two encoders
            for nm in ("pretrained", "scratch"):
                F = FEATS[nm]
                t0 = time.time(); clf = fit_probe(F[tr_rows][sub], y_tr[sub], C_STAR[nm]); fit_s = time.time() - t0
                yp = clf.predict(F[te_rows])
                sc = score_pack(y_te, yp)
                rows.append(dict(encoder=nm, fraction=frac, seed=r, C=C_STAR[nm], n_train=int(len(sub)),
                                 fit_s=round(fit_s, 4), **{k: round(v, 4) for k, v in sc.items()}))
                if r == 0: preds0[(nm, frac)] = yp
                if abs(frac - 1.0) < 1e-9 and r == 0:
                    joblib.dump(dict(clf=clf, encoder=nm, C=C_STAR[nm], feature_dim=int(F.shape[1]), frozen=True,
                                     note="probe head for the 100 %-labels run; encoder weights live in models/mae_encoder.pth (pretrained) or are re-seeded (scratch)"),
                                f"models/probe_{nm}.joblib")
RUNS = pd.DataFrame(rows)
RUNS.to_csv("reports/probe_runs.csv", index=False)
_agg = RUNS.groupby(["encoder", "fraction"]).agg(n=("accuracy", "size"), acc_mean=("accuracy", "mean"), acc_sd=("accuracy", "std"),
                                                 f1_mean=("macro_f1", "mean"), f1_sd=("macro_f1", "std")).reset_index()
_rows = [[r_["encoder"], f"{r_['fraction']*100:.0f} %", f"{int(r_['n'])}", f"{r_['acc_mean']:.3f} ± {r_['acc_sd']:.3f}",
          f"{r_['f1_mean']:.3f} ± {r_['f1_sd']:.3f}"] for _, r_ in _agg.iterrows()]
hl = next((i for i, r_ in enumerate(_rows) if r_[0] == "pretrained" and r_[1] == "100 %"), None)
html_table(_rows, ["encoder", "label fraction", "runs", "overall accuracy (mean ± sd over seeds)", "macro F1 (mean ± sd)"], highlight=hl, width="75%")
ok(f"Paired probe runs done: {len(RUNS)} fits · every subset reused by both encoders · per-run numbers in reports/probe_runs.csv")

## Block 7 · Evaluation: learning curves & paired block-bootstrap CIs
**What it does:** scores both probes on the held-out test blocks at every label fraction; resamples **whole test blocks** with replacement — the same draws for both encoders, so the difference is **paired** — to get 95 % CIs for each score and for the pretrained-minus-scratch macro-F1 difference; draws the learning curves and the confusion matrices at 100 % of labels; writes `reports/metrics.json`. Macro-F1 is computed over all 7 classes (a class absent from the test set scores 0, for both encoders alike).

**Why it matters:** a single number can always be luck. The block bootstrap propagates the spatial clustering of the test set, and the paired difference CI is the honest way to say whether pretraining helped — an interval that includes 0 means **this run cannot separate the two encoders**.

> 🎤 **Say to the panel:** *My headline is the paired difference with its confidence interval — not two separate scores. If the interval spans zero, I say the run cannot separate them, and I do not dress that up as a win.*

In [ ]:
banner("7", "Evaluation on held-out test blocks", f"block-bootstrap 95 % CIs ({CFG['boot_n']} resamples) · paired pretrained-minus-scratch differences · macro F1 over all 7 classes")
def metrics_from_cm(cm):
    """All headline metrics from a 7x7 confusion matrix (rows = reference, columns = prediction)."""
    cm = np.asarray(cm, dtype=np.float64)
    tp, sup, prd, n = np.diag(cm), cm.sum(1), cm.sum(0), cm.sum()
    pr = np.divide(tp, prd, out=np.zeros_like(tp), where=prd > 0)
    rc = np.divide(tp, sup, out=np.zeros_like(tp), where=sup > 0)
    f1 = np.divide(2 * pr * rc, pr + rc, out=np.zeros_like(tp), where=(pr + rc) > 0)
    acc = tp.sum() / n if n else 0.0
    pe = float((sup * prd).sum() / n ** 2) if n else 0.0
    kappa = (acc - pe) / (1 - pe) if pe < 1 else 0.0
    return dict(accuracy=float(acc), macro_f1=float(f1.mean()), kappa=float(kappa),
                per_class={CLASS_NAMES[i]: dict(f1=float(f1[i]), support=int(sup[i])) for i in range(7)})

def block_cms(y, p, blk):
    """Per-spatial-block confusion matrices -> array (n_blocks, 7, 7)."""
    ub, inv = np.unique(blk, return_inverse=True)
    flat = inv.astype(np.int64) * 49 + y.astype(np.int64) * 7 + p.astype(np.int64)
    return np.bincount(flat, minlength=len(ub) * 49).reshape(len(ub), 7, 7)

def block_bootstrap(cms_by_model, n_boot, seed):
    """Resample whole test blocks with replacement. The SAME draws are used for every model, so differences are paired."""
    rng = np.random.default_rng(seed); nb_ = next(iter(cms_by_model.values())).shape[0]
    keys = ["accuracy", "macro_f1", "kappa"]
    draws = {m: {k: [] for k in keys} for m in cms_by_model}
    for _ in range(n_boot):
        w = np.bincount(rng.integers(0, nb_, nb_), minlength=nb_)          # how often each block is drawn
        for m, cms in cms_by_model.items():
            r = metrics_from_cm(np.tensordot(w, cms, axes=1))
            for k in keys: draws[m][k].append(r[k])
    return {m: {k: np.asarray(v) for k, v in d.items()} for m, d in draws.items()}

def ci(a, lo=2.5, hi=97.5):
    a = a[np.isfinite(a)]
    return (float("nan"),) * 3 if len(a) < 20 else (float(np.mean(a)), float(np.percentile(a, lo)), float(np.percentile(a, hi)))

te_blk = (DS_ORIGINS[te_rows][:, 0] // BP) * nBC + (DS_ORIGINS[te_rows][:, 1] // BP)
n_test_blocks = int(len(np.unique(te_blk)))
if n_test_blocks < 10:
    warn(f"Only <b>{n_test_blocks}</b> test blocks: bootstrap intervals are wide and unstable, and resampling so few blocks understates the true uncertainty.")

FULL, CIP, DIFF, VERDICT = {}, {}, {}, {}
for frac in CFG["fractions"]:
    cms = {nm: block_cms(y_te, preds0[(nm, frac)], te_blk) for nm in ("pretrained", "scratch")}
    FULL[frac] = {nm: metrics_from_cm(cm.sum(0)) for nm, cm in cms.items()}   # full-test CM = sum of per-block CMs
    boot = block_bootstrap(cms, CFG["boot_n"], SEED + 31)
    CIP[frac] = {nm: {k: ci(v) for k, v in d.items()} for nm, d in boot.items()}
    d_ = boot["pretrained"]["macro_f1"] - boot["scratch"]["macro_f1"]
    d_ = d_[np.isfinite(d_)]
    pt = FULL[frac]["pretrained"]["macro_f1"] - FULL[frac]["scratch"]["macro_f1"]
    DIFF[frac] = (float(pt), float(np.percentile(d_, 2.5)), float(np.percentile(d_, 97.5))) if len(d_) >= 20 else (float(pt), float("nan"), float("nan"))
    lo, hi = DIFF[frac][1], DIFF[frac][2]
    if not np.isfinite(lo): VERDICT[frac] = "CI unavailable (too few test blocks)"
    elif lo <= 0 <= hi: VERDICT[frac] = "no detectable difference (95 % CI spans 0)"
    else: VERDICT[frac] = "pretrained better" if pt > 0 else "scratch better"

_f = lambda v: f"{v:.3f}"
def _fci(frac, nm, k):
    v = FULL[frac][nm][k]; _, lo, hi = CIP[frac][nm][k]
    return _f(v) if not np.isfinite(lo) else f"{_f(v)} [{lo:.3f}, {hi:.3f}]"
_rows = [[f"{frac*100:.0f} %", _fci(frac, "pretrained", "macro_f1"), _fci(frac, "scratch", "macro_f1"),
          f"{DIFF[frac][0]:+.3f}" + ("" if not np.isfinite(DIFF[frac][1]) else f" [{DIFF[frac][1]:+.3f}, {DIFF[frac][2]:+.3f}]"),
          VERDICT[frac]] for frac in CFG["fractions"]]
hl = next((i for i, r_ in enumerate(_rows) if r_[0] == "100 %"), None)
html_table(_rows, ["label fraction", "pretrained macro F1 [95 % CI]", "scratch macro F1 [95 % CI]",
                   "Δ (pretrained − scratch) [95 % CI]", "verdict (computed, not claimed)"], highlight=hl, width="95%")
info(f"<b>Paired comparison on {n_test_blocks} test blocks ({CFG['boot_n']} bootstrap resamples of whole blocks):</b> "
     "an interval that contains 0 means this run cannot separate the two encoders - absence of evidence is not evidence of absence, "
     "especially under a time-boxed pretraining budget.")

# ---- figure: learning curves with bootstrap CIs ----
fig, axs = plt.subplots(1, 2, figsize=(13.5, 4.8))
for axm, k_, ttl in ((axs[0], "accuracy", "Overall accuracy"), (axs[1], "macro_f1", "Macro F1 (all 7 classes)")):
    xs = np.arange(len(CFG["fractions"]))
    for off, (nm, c_) in ((-0.045, ("pretrained", "#2a9d8f")), (0.045, ("scratch", "#e76f51"))):
        pts = np.array([FULL[fr][nm][k_] for fr in CFG["fractions"]])
        lo = np.array([CIP[fr][nm][k_][1] for fr in CFG["fractions"]]); hi = np.array([CIP[fr][nm][k_][2] for fr in CFG["fractions"]])
        ok_y = np.isfinite(lo) & np.isfinite(hi)
        axm.errorbar(xs[ok_y] + off, pts[ok_y], yerr=[pts[ok_y] - lo[ok_y], hi[ok_y] - pts[ok_y]],
                     fmt="o-", color=c_, capsize=4, label=f"{nm} encoder (95 % block-bootstrap CI)")
        axm.plot(xs[~ok_y] + off, pts[~ok_y], "o", color=c_)
    axm.set_xticks(xs); axm.set_xticklabels([f"{f*100:.0f} %" for f in CFG["fractions"]]); axm.set_xlabel("share of labelled patches")
    axm.set_ylabel(k_.replace("_", " ")); axm.set_ylim(0, 1); axm.grid(alpha=.3); axm.legend(fontsize=8.5)
    axm.set_title(ttl, fontweight="bold")
fig.suptitle("Frozen-encoder probe on held-out spatial blocks · error bars = block-bootstrap 95 % CI (seed variability in reports/probe_runs.csv)", fontweight="bold", fontsize=10.5)
plt.tight_layout(); plt.savefig("reports/06_learning_curves.png", dpi=130); plt.show()

# ---- figure: confusion matrices + per-class F1 at 100 % of labels ----
def cm_norm(cm):
    m = np.asarray(cm, float); return m / np.maximum(m.sum(1, keepdims=True), 1)
fig, axs = plt.subplots(1, 3, figsize=(19, 5.4), gridspec_kw=dict(width_ratios=[1, 1, 1.05]))
for axm, (nm, ttl) in zip(axs[:2], (("pretrained", "MAE-pretrained encoder"), ("scratch", "Scratch encoder"))):
    m = cm_norm(block_cms(y_te, preds0[(nm, 1.0)], te_blk).sum(0))
    axm.imshow(m, cmap="Blues", vmin=0, vmax=1)
    for i in range(7):
        for j in range(7): axm.text(j, i, f"{m[i, j]:.2f}", ha="center", va="center", fontsize=7.5, color="white" if m[i, j] > .5 else "black")
    axm.set_xticks(range(7)); axm.set_yticks(range(7))
    axm.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=7.5); axm.set_yticklabels(CLASS_NAMES, fontsize=7.5)
    axm.set_xlabel("predicted"); axm.set_ylabel("reference"); axm.set_title(f"{ttl} · 100 % labels · row-normalised", fontweight="bold", fontsize=10)
wd = .38
axs[2].bar(np.arange(7) - wd / 2, [FULL[1.0]["pretrained"]["per_class"][n_]["f1"] for n_ in CLASS_NAMES], wd, label="pretrained", color="#2a9d8f")
axs[2].bar(np.arange(7) + wd / 2, [FULL[1.0]["scratch"]["per_class"][n_]["f1"] for n_ in CLASS_NAMES], wd, label="scratch", color="#e76f51")
axs[2].set_xticks(np.arange(7)); axs[2].set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=7.5)
axs[2].set_ylim(0, 1.18); axs[2].set_ylabel("F1"); axs[2].legend(); axs[2].grid(axis="y", alpha=.3)
axs[2].set_title("Per-class F1 at 100 % labels", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.savefig("reports/07_confusion_100.png", dpi=130); plt.show()

# ---- save every number computed in this run ----
_results = {}
for frac in CFG["fractions"]:
    _results[str(frac)] = dict(
        pretrained=_py(FULL[frac]["pretrained"]), scratch=_py(FULL[frac]["scratch"]),
        ci95_block_bootstrap=_py(CIP[frac]),
        delta_macro_f1_pretrained_minus_scratch=_py(DIFF[frac]), verdict=VERDICT[frac])
METRICS = dict(
    mode=MODE, seed=SEED, device=str(DEVICE), torch_version=torch.__version__,
    config=_py({k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}),
    pretraining=_py(PRETRAIN),
    data=dict(label_source=scheme,
              label_note="ESA WorldCover used as PROXY labels - every score is agreement with WorldCover, not field accuracy; class 4 is herbaceous wetland (peat proxy)",
              n_ssl_patches=int(len(SSL_PATCHES)), n_lab_patches=int(len(DS_PATCHES)),
              purity_mean=float(DS_PURITY.mean()) if len(DS_PURITY) else None,
              split_seed=int(chosen_seed), block_px=int(BP * CFG["patch"]),
              blocks=dict(train=int((BS == 0).sum()), val=int((BS == 1).sum()), test=int((BS == 2).sum())),
              class_counts_per_split={_split_names[k]: dict(zip(CLASS_NAMES, split_counts[k])) for k in (0, 1, 2)}),
    probe=dict(frozen_encoder=True, feature_dim=int(F_PRE.shape[1]), C_star=_py(C_STAR),
               probe="LogisticRegression (lbfgs, class_weight=balanced), C chosen on validation blocks",
               macro_f1_scope="over all 7 classes, absent classes score 0 (same rule for both encoders)",
               n_test_blocks=n_test_blocks, bootstrap_resamples=int(CFG["boot_n"]), probe_seeds=int(CFG["probe_seeds"]),
               runs=rows),
    results=_py(_results),
    honesty=dict(claims="No improvement is claimed: the paired block-bootstrap CI of the macro-F1 difference decides, and spans of 0 are reported as 'no detectable difference'.",
                 scope="One small region, one season, time-boxed pretraining, frozen encoder + linear probe. Fine-tuning the full encoder is NOT tested here and could change the picture."),
    timings_s={k: round(v, 1) for k, v in TIMINGS.items()})
json.dump(_py(METRICS), open("reports/metrics.json", "w"), indent=2)
ok("Saved: reports/metrics.json · reports/probe_runs.csv · reports/06_learning_curves.png · reports/07_confusion_100.png")

## Block 8 · Run summary, honest limitations & export
**What it does:** prints stage timings and a verdict that is **computed from the confidence intervals**, writes the honest-limitations note and `reports/run_info.json` (environment, exact scene IDs, whether the pretraining budget was hit), zips `reports/` and offers the download.

**Why it matters:** for a public repository, what the run did *not* establish matters as much as what it did. Everything printed below is derived from this run's own numbers — nothing is typed in by hand.

> 🎤 **Say to the panel:** *I can tell you exactly what this run does and does not show: it measures pretraining effects with a paired CI on proxy labels, in one region and season, with a frozen encoder — and it claims nothing beyond that.*

In [ ]:
banner("8", "Run summary & export", "everything below is computed from this run - nothing is typed in by hand")
total = sum(TIMINGS.values())
html_table([[k.replace("_", " ").title(), f"{v:.0f} s", f"{v/total*100:.0f} %"] for k, v in TIMINGS.items()] +
           [["<b>total (tracked stages)</b>", f"<b>{total:.0f} s ({total/60:.1f} min)</b>", ""]],
           ["stage", "time", "share"], width="45%")
warn("Labels are a <b>PROXY</b> (ESA WorldCover): every score is agreement with WorldCover, not field-verified accuracy. State this clearly if you show these numbers.")
if MODE == "demo": warn("<b>DEMO MODE</b> - reduced patch counts and a short pretraining budget for a fast run. Do not quote these numbers as final results; set MODE = \"full\" in Block 1 for the reported run.")
if PRETRAIN["budget_hit"]: warn("The <b>MAE time budget was hit</b> - the comparison is time-boxed, not an architecture verdict.")
info("<b>Read these as honest limitations:</b> "
     f"pretraining used {len(SSL_PATCHES):,} unlabelled patches over {len(SSL_AREAS)} small areas and ran for {PRETRAIN['seconds']:.0f} s ({PRETRAIN['stop_reason']}); "
     f"the encoder was frozen (mean-pooled tokens + logistic probe), so this measures linear-probe quality only - full fine-tuning is NOT tested and could change the picture; "
     f"labels are ESA {scheme} (a modelled map, no field validation; class 4 is herbaceous wetland standing in for peat); "
     f"one labelled region and one season; {n_test_blocks} test blocks (CIs reflect test-block sampling only, not training randomness or other splits, and neighbouring blocks are correlated); "
     "macro F1 counts absent classes as 0 for both encoders alike; MAE was chosen over contrastive learning for small-batch friendliness, and no colour jitter was used to keep spectra honest.")
_pt, _lo, _hi = DIFF[1.0]
if not np.isfinite(_lo): _verdict = "confidence interval unavailable (too few test blocks)"
elif _lo <= 0 <= _hi: _verdict = "no detectable difference (95 % CI of the paired macro-F1 difference spans 0)"
else: _verdict = "pretrained encoder better" if _pt > 0 else "scratch encoder better"
info(f"<b>Headline (100 % of labels, macro F1):</b> pretrained {_fci(1.0, 'pretrained', 'macro_f1')} vs scratch {_fci(1.0, 'scratch', 'macro_f1')} → {_verdict}. "
     f"Δ = {_pt:+.3f} [{_lo:+.3f}, {_hi:+.3f}] on {n_test_blocks} test blocks. "
     "If the interval spans 0, this run cannot separate the two encoders - and this notebook does not claim otherwise.")

# ---- honest README note (auto-generated) ----
_lines = [
    "# sentinel2-self-supervised · run note",
    "",
    f"- Mode: **{MODE}** · seed {SEED} · device {DEVICE} · torch {torch.__version__}",
    f"- Pretraining: MAE (dim {CFG['mae_dim']}, depth {CFG['mae_depth']}, mask {CFG['mae_mask']*100:.0f} %) on {len(SSL_PATCHES):,} unlabelled 64×64 Sentinel-2 patches "
    f"({', '.join(a_['name'] for a_ in SSL_AREAS)}) · {PRETRAIN['epochs_run']} epochs · best val masked-MSE {PRETRAIN['best_val_loss']:.4f} · {PRETRAIN['stop_reason']}",
    f"- Downstream: {len(DS_PATCHES):,} labelled patches ({scheme} PROXY labels, mean purity {RUN_INFO['ds']['purity_mean']*100:.1f} %) over {DS_AREA['name']}; "
    f"spatial split seed {chosen_seed} ({int((BS==0).sum())}/{int((BS==1).sum())}/{int((BS==2).sum())} blocks, buffer {CFG['buffer_blocks']} block)",
    f"- Probe: frozen encoder → mean token pooling → logistic regression (C={C_STAR['pretrained']} pretrained / C={C_STAR['scratch']} scratch, chosen on validation blocks)",
    f"- Headline at 100 % labels: macro F1 {_fci(1.0, 'pretrained', 'macro_f1')} (pretrained) vs {_fci(1.0, 'scratch', 'macro_f1')} (scratch) · "
    f"Δ {_pt:+.3f} [{_lo:+.3f}, {_hi:+.3f}] → {_verdict}",
    "",
    "## Honest limitations",
    "- Labels are WorldCover (modelled, not field truth); class 4 is herbaceous wetland used as a peat PROXY.",
    "- One region + season for evaluation; small areas; time-boxed pretraining; frozen encoder only (no fine-tuning test).",
    "- No improvement is claimed unless the paired CI excludes 0 in the direction of the pretrained encoder.",
    "- Macro F1 is computed over all 7 classes; absent classes contribute 0 for both encoders alike.",
    "",
    "## Files",
    "- 01_ssl_patches.png · 02_masking.png · 03_reconstructions.png · 04_pretrain_curve.png",
    "- 05_labelled_area.png · 06_learning_curves.png · 07_confusion_100.png",
    "- pretrain_log.csv · probe_runs.csv · data_quality.json · metrics.json · run_info.json",
]
Path("reports/README_note.md").write_text("\n".join(_lines), encoding="utf-8")
json.dump(_py(dict(mode=MODE, seed=SEED, python=sys.version.split()[0], torch=torch.__version__, device=str(DEVICE),
                   deterministic_requested=True, timings_s={k: round(v, 1) for k, v in TIMINGS.items()},
                   pretraining=_py(PRETRAIN), ssl_areas=RUN_INFO["ssl_areas"], ds=RUN_INFO["ds"],
                   label_source=scheme, split_seed=int(chosen_seed), n_test_blocks=n_test_blocks,
                   bootstrap_resamples=int(CFG["boot_n"]),
                   headline=dict(metric="macro_f1", fraction=1.0,
                                 pretrained_ci95=list(CIP[1.0]["pretrained"]["macro_f1"][1:]),
                                 scratch_ci95=list(CIP[1.0]["scratch"]["macro_f1"][1:]),
                                 delta=list(DIFF[1.0]), verdict=_verdict))),
          open("reports/run_info.json", "w"), indent=2)

shutil.make_archive("sentinel2_self_supervised_reports", "zip", "reports")
ok("Saved: " + " · ".join(sorted(p.name for p in Path("reports").iterdir())) +
   " · models/mae_encoder.pth · models/probe_pretrained.joblib · models/probe_scratch.joblib · sentinel2_self_supervised_reports.zip")
try:
    from google.colab import files as _f; _f.download("sentinel2_self_supervised_reports.zip")
except Exception:
    pass